# PhysAR-IDS -- NB7b (v6): how much does the gate protect, and what does it cost in false alarms?

**Why.** NB6 / NB6b showed that a fully adaptive white-box attacker cannot evade the pipeline on ransomware or reverseshell. NB7 then showed that this protection sits at an operating point with an 11% false-positive rate (the threshold was calibrated on out-of-fold errors that leak: identical copies of every held-out row were in the training folds). With a time-blocked, properly calibrated gate (3.6% false positives) a gradient attack on the gate alone evades ransomware and reverseshell about 80% of the time at 1.0 sigma. So the NB6 statement "the gate stops ransomware and reverseshell" is a trade-off, and this notebook measures it.

**What is run.** For each of five thresholds of the gate -- the 90th, 95th (= NB6), 97.5th and 99th percentile of the NB6 gate's out-of-fold benign error, plus the NB7 time-blocked gate (`all_blk`, 95th percentile of its own blocked folds) -- the **fully adaptive white-box** of NB6 (maximise `min(margin gap, (tau - error) / tau)`; final decisions by the CPU classifier and the gate) is run against the NB5b robust model at 1.0 sigma on the cyber features. A gate-unaware white-box attack on Layer 1 is run once per seed and its outputs are scored against each threshold.

**Reported per threshold.** Test false-positive rate of the gate and of the whole pipeline (benign rows that Layer 1 or the gate flags), clean attack detection, and the pipeline evasion of the gate-unaware and the fully adaptive attacker, overall and per class. **Read it as** a curve of protection against false alarms; a threshold that keeps ransomware / reverseshell evasion at 0.000 with low false alarms would be the recommended setting, and if none does, that is the finding. The search is greedy, so evasion rates are lower bounds.

**Needs:** `ae_gate_v6_seed{s}.pt` (NB6), `ae_gate7_all_blk_v6_seed{s}.pt` (NB7), `xgb_robust_v6_seed{s}.json` (NB5b), `attack3_seed{s}_v6.pkl` (NB6) and the data files. **Runtime:** about 3 h (about 35 min per seed).

## 0. Setup

In [ ]:
import os, sys, random, time, json, hashlib, shutil, platform, warnings, copy
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score
import matplotlib; import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
try:
    from sb3_contrib import RecurrentPPO
    from stable_baselines3.common.callbacks import CheckpointCallback
except ImportError as e:
    raise ImportError('Install in a terminal (not in a cell): pip install stable-baselines3 sb3-contrib gymnasium --no-deps') from e
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False

# ---- PPO / environment settings (unchanged from the reference design) ----
EPS_BOUND = 0.3; MAX_STEPS = 10
EVASION_REWARD = 10.0; STEP_PENALTY_SCALE = 0.01; TIMEOUT_PENALTY = -1.0
TOTAL_TIMESTEPS = 50000; CKPT_FREQ = 5000
EPS_TOTAL_BUDGET = 0.1
CLIP_BUDGETS = {'1.0': 1.0, 'unconstrained': None}     # budgets at which agents are RETRAINED with range clipping (stage B)
SAVED_AGENTS = {'0.1': ('ppo_attack2_budgeted_{v}_seed{s}', 0.1), '0.2': ('ppo_attack2_budget0p2_{v}_seed{s}', 0.2), '0.3': ('ppo_attack2_budget0p3_{v}_seed{s}', 0.3),
                '0.5': ('ppo_attack2_budget0p5_{v}_seed{s}', 0.5), '1.0': ('ppo_attack2_budget1p0_{v}_seed{s}', 1.0), 'unconstrained': ('ppo_attack2_unconstrained_{v}_seed{s}', None)}
EVAL_CHUNK = 8192


# ---- Defence 1 (adversarial training) settings ----
PGD_EPS = 0.1; PGD_ITERS = 10                       # as NB3 / NB3b (all 93 features, alpha = eps / 4)
TRAIN_PPO_TAGS = ['0.1', '0.3']                     # PPO agents (NB4 / NB4b, trained against the STANDARD model) whose examples go into the training set
EVAL_PPO_TAGS = ['0.1', '0.3', '0.5', '1.0', 'unconstrained']   # budgets 0.5 / 1.0 / unconstrained were never seen in training
N_PPO_TRAIN_PER_CLASS = 8000
AUG_RATIO = 1.0                                     # adversarial training rows per clean row (the paper's 1:1)
# ---- white-box (tree-greedy, NB3c / NB3d) attack against both models ----
K_CAND = 12; MAX_ITERS = 60; CHUNK = 20000; BLOCK = 300_000
WB_BUDGETS = {'0.1': 0.1, '0.3': 0.3, '0.5': 0.5, '1.0': 1.0}
N_WB_PER_CLASS = 2000                               # attack rows per class searched (same rows for every seed, model and budget)


# ---- NB6: Attack 3 -- adaptive attacker against the WHOLE pipeline (Layer 1 classifier + Layer 2 autoencoder gate) ----
LAYER1 = {'rob': 'xgb_robust_{v}_seed{s}.json', 'def': 'xgb_defended_{v}_seed{s}.json'}      # NB5b robust model (primary) and NB5 defended model
AW_BUDGETS = {'0.1': 0.1, '0.3': 0.3, '1.0': 1.0}      # sigma, cyber features only (same attacker-controlled set as NB3c / NB5)
N_AW_PER_CLASS = 1000                                    # rows per attack class (a random subset of NB5's 2,000 per class)
MAX_ITERS_AWARE = 100                                    # rounds for the greedy search against the pipeline
GATE_SCALE = 1.0                                         # weight of the gate term in the joint score min(margin gap, GATE_SCALE * (tau - err) / tau)
N_FOLDS = 5; GATE_EPOCHS = 30; GATE_PCTL = 95; GATE_HIDDEN = 32; GATE_BOTTLENECK = 12; GATE_CYBER_BOOST = 1.0
RUN_PPO_AWARE = True; PPO_AWARE_MODELS = ['rob']; PPO_AWARE_BUDGETS = {'0.3': 0.3, '1.0': 1.0}      # black-box PPO retrained against the pipeline's hard-label verdict
PPO_UNAWARE_TAGS = ['0.3', '1.0', 'unconstrained']      # the saved NB4 agents (trained against the standard classifier, gate-unaware)


# ---- NB7b: protection vs false alarms -- the fully adaptive white-box attack against the pipeline at several gate thresholds ----
L1_USE = ['rob']                        # Layer-1 model(s): the NB5b robust model (primary)
GATE_PCTLS_B = [90, 95, 97.5, 99]       # threshold = this percentile of the NB6 gate's out-of-fold benign error (95 = NB6)
INCLUDE_BLK = True                      # plus the NB7 time-blocked gate (all_blk, its own 95th-percentile threshold)
ADAPT_ITERS = 150                       # NB6b: no row was still improving after 147-263 rounds; the search stops by itself when no row can improve
ADAPT_BUDGETS = {'1.0': 1.0}            # sigma, cyber features only
N_ADAPT_PER_CLASS = 300                 # random subset of NB6's 1,000 rows per class (same rows for every seed, threshold and model)

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
BACKEND_NAME = 'cpu(hist)' if _cpu else 'gpu(cuda,hist)'
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} fit backend: {BACKEND_NAME}')
import sb3_contrib, stable_baselines3
print(f'sb3-contrib {sb3_contrib.__version__} | stable-baselines3 {stable_baselines3.__version__} | gymnasium {gym.__version__}')
print('Working directory:', os.getcwd())

## 1. Load v6 data

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; N_CYBER = len(CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)

atk_tr = y_train != 0; X_atk_train, y_atk_train = X_train[atk_tr], y_train[atk_tr]
atk_te = y_test != 0;  X_atk_test,  y_atk_test  = X_test[atk_te],  y_test[atk_te]
test_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0][atk_te]
FEAT_LO, FEAT_HI = X_train.min(0), X_train.max(0)
print(f'Features {N_FEAT} | attacker-controlled (cyber) {N_CYBER} | training pool {X_atk_train.shape} | evaluation rows {X_atk_test.shape} in {len(np.unique(test_burst))} attack bursts')
print('Training pool class counts:', np.bincount(y_atk_train)[1:], '| evaluation class counts:', np.bincount(y_atk_test)[1:])
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1):.4f}')

## 2. White-box attack

In [ ]:
def get_thresholds(clf):
    """feature index -> sorted unique float32 split thresholds used anywhere in the ensemble."""
    df = clf.get_booster().trees_to_dataframe(); sp = df[df['Feature'] != 'Leaf']
    return {int(str(f).lstrip('f')): np.unique(g['Split'].to_numpy(dtype=np.float32)) for f, g in sp.groupby('Feature')}

class Margin:
    """Raw per-class scores of a fitted classifier (the white-box access). Scores in blocks and, if the GPU runs out of memory, halves the block and retries."""
    def __init__(self, clf): self.b = clf.get_booster(); self.bs = BLOCK
    def _pred(self, X):
        while True:
            try:
                return [np.asarray(self.b.inplace_predict(np.ascontiguousarray(X[s:s + self.bs], dtype=np.float32), predict_type='margin', validate_features=False)) for s in range(0, len(X), self.bs)]
            except xgb.core.XGBoostError as e:
                if 'out of memory' not in str(e).lower() or self.bs <= 20000: raise
                self.bs = max(20000, self.bs // 2); print(f'      GPU out of memory -> scoring in blocks of {self.bs:,} rows', flush=True)
    def __call__(self, X):
        out = self._pred(X); return np.concatenate(out) if len(out) > 1 else out[0]
def gap_of(z): return (z[:, 0] - z[:, 1:].max(1)).astype(np.float64)       # > 0  <=>  benign is the arg-max class

def interval(o, j, B):
    """allowed interval of feature j for rows with original values o: L-inf ball (if B) intersected with the training range (never further out than the row already is)."""
    lo_r, hi_r = np.minimum(FEAT_LO[j], o), np.maximum(FEAT_HI[j], o)
    if B is None: return lo_r.astype(np.float32), hi_r.astype(np.float32)
    return np.maximum(o - np.float32(B), lo_r).astype(np.float32), np.minimum(o + np.float32(B), hi_r).astype(np.float32)

def candidates(xc, X0c, j, T, B, K):
    """candidate (row, value) moves for feature j: the K nearest thresholds on each side inside the allowed interval, plus the interval ends."""
    xv = xc[:, j]; a, b = interval(X0c[:, j], j, B); nT = len(T); idx = np.searchsorted(T, xv, side='right'); R, V = [], []
    for d in range(min(K, nT)):
        k = idx + d; v = T[np.minimum(k, nT - 1)]; ok = (k < nT) & (v <= b); R.append(np.flatnonzero(ok)); V.append(v[ok])
        k = idx - 1 - d; v = np.nextafter(T[np.maximum(k, 0)], np.float32(-np.inf)); ok = (k >= 0) & (v >= a); R.append(np.flatnonzero(ok)); V.append(v[ok])
    ok = b > xv; R.append(np.flatnonzero(ok)); V.append(b[ok]); ok = a < xv; R.append(np.flatnonzero(ok)); V.append(a[ok])
    return np.concatenate(R), np.concatenate(V).astype(np.float32)

def one_step(margin, xc, X0c, g, feats, TH, B, K):
    """best single-feature move per row (by the model's own margin gap). Returns best_gap, best_feature (-1 = none helps), best_value, n_scored."""
    n = len(xc); best_g = g.copy(); best_j = np.full(n, -1, np.int64); best_v = np.zeros(n, np.float32); n_scored = 0
    bufR, bufJ, bufV = [], [], []; size = 0
    def flush():
        nonlocal n_scored
        if not bufR: return
        R, J, V = np.concatenate(bufR), np.concatenate(bufJ), np.concatenate(bufV); bufR.clear(); bufJ.clear(); bufV.clear()
        for s in range(0, len(R), BLOCK):
            r, jj, v = R[s:s + BLOCK], J[s:s + BLOCK], V[s:s + BLOCK]
            Xc = xc[r]; Xc[np.arange(len(r)), jj] = v                    # xc[r] is a copy
            gg = gap_of(margin(Xc)); n_scored += len(r)
            o = np.argsort(r, kind='stable'); rs, gs = r[o], gg[o]
            starts = np.flatnonzero(np.r_[True, rs[1:] != rs[:-1]]); segmax = np.maximum.reduceat(gs, starts); cnt = np.diff(np.r_[starts, len(rs)])
            hit = np.flatnonzero(gs == np.repeat(segmax, cnt)); _, first = np.unique(rs[hit], return_index=True); sel = o[hit[first]]
            rows = r[sel]; better = gg[sel] > best_g[rows]; rr = rows[better]
            best_g[rr] = gg[sel][better]; best_j[rr] = jj[sel][better]; best_v[rr] = v[sel][better]
    for j in feats:
        T = TH.get(int(j))
        if T is None: continue
        R, V = candidates(xc, X0c, int(j), T, B, K)
        if len(R) == 0: continue
        bufR.append(R); bufJ.append(np.full(len(R), j, np.int64)); bufV.append(V); size += len(R)
        if size >= BLOCK: flush(); size = 0
    flush(); return best_g, best_j, best_v, n_scored

def whitebox_trace(margin, X0, y0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):
    """as NB3c's whitebox, plus a trace: per round, the evaded share per class (classes 4, 5) and the median margin gap of the rows not yet evaded."""
    x = X0.copy(); g = gap_of(margin(x)); active = g <= 0; stalled = np.zeros(len(x), bool); trace = []; t_start = time.time()
    active = ~(g > 0); init_ev = g > 0
    for it in range(max_iters):
        act = np.flatnonzero(active)
        if len(act) == 0: break
        t0 = time.time(); scored = 0
        for s in range(0, len(act), chunk):
            ids = act[s:s + chunk]; bg, bj, bv, ns = one_step(margin, x[ids], X0[ids], g[ids], feats, TH, B, K); scored += ns
            imp = bj >= 0; ii = ids[imp]; x[ii, bj[imp]] = bv[imp]; g[ii] = bg[imp]; stalled[ids[~imp]] = True; active[ids[~imp]] = False
        active &= (g <= 0); ev = g > 0
        trace.append({'round': it + 1, 'active': int(len(act)), 'evaded_all': float(ev.mean()), 'evaded_pc': [float(ev[y0 == c].mean()) for c in range(1, 6)], 'evaded_c4': float(ev[y0 == 4].mean()), 'evaded_c5': float(ev[y0 == 5].mean()),
                      'median_gap_unevaded': float(np.median(g[~ev])) if (~ev).any() else float('nan'), 'seconds': time.time() - t0})
        if verbose and ((it + 1) % 10 == 0 or it == 0): print(f'      round {it + 1:>3}: active {len(act):>5} | evaded ransomware {trace[-1]["evaded_c4"]:.3f} reverseshell {trace[-1]["evaded_c5"]:.3f} | median gap of unevaded {trace[-1]["median_gap_unevaded"]:+.3f} | {time.time() - t0:4.1f}s', flush=True)
    return x, g, init_ev, stalled, trace
def whitebox(margin, X0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):      # used by the self-test
    return whitebox_trace(margin, X0, np.zeros(len(X0), dtype=np.int64), feats, TH, B, K, max_iters, chunk, verbose)
print('white-box attack defined')

## 3. Gate and joint score (NB6 code)

In [ ]:
class AE(nn.Module):
    def __init__(self, n_features, bottleneck=GATE_BOTTLENECK, hidden=GATE_HIDDEN):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_features, hidden), nn.GELU(), nn.Linear(hidden, bottleneck), nn.GELU())
        self.decoder = nn.Sequential(nn.Linear(bottleneck, hidden), nn.GELU(), nn.Linear(hidden, n_features))
    def forward(self, x): return self.decoder(self.encoder(x))
FW = torch.ones(N_FEAT, device=DEVICE); FW[torch.as_tensor(CYBER_IDX, device=DEVICE)] = GATE_CYBER_BOOST
def wmse(recon, target): return (((recon - target) ** 2) * FW).sum(1) / FW.sum()

class Gate:
    def __init__(self, models, tau): self.models = models; self.tau = float(tau)
    @torch.no_grad()
    def score(self, X, bs=65536):
        X = np.ascontiguousarray(X, dtype=np.float32); out = np.empty(len(X), np.float64)
        for s in range(0, len(X), bs):
            b = torch.from_numpy(X[s:s + bs]).to(DEVICE); out[s:s + bs] = torch.stack([wmse(m(b), b) for m in self.models]).mean(0).double().cpu().numpy()
        return out
    def flag(self, X): return self.score(X) > self.tau

def train_gate(seed, X_benign):
    from sklearn.model_selection import KFold
    oof = np.zeros(len(X_benign)); models = []
    for k, (tr, ho) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=seed).split(X_benign)):
        torch.manual_seed(seed * 10 + k); m = AE(N_FEAT).to(DEVICE); opt = torch.optim.Adam(m.parameters(), lr=1e-3)
        ds = TensorDataset(torch.from_numpy(X_benign[tr]).float()); loader = DataLoader(ds, batch_size=256, shuffle=True, generator=torch.Generator().manual_seed(seed * 10 + k))
        for ep in range(GATE_EPOCHS):
            m.train()
            for (xb,) in loader: xb = xb.to(DEVICE); opt.zero_grad(); wmse(m(xb), xb).mean().backward(); opt.step()
        m.eval(); models.append(m); oof[ho] = Gate([m], 0).score(X_benign[ho])
    return Gate(models, np.percentile(oof, GATE_PCTL)), oof

class PipelineOracle:
    """hard-label verdict of the whole pipeline: a row Layer 1 calls benign but the gate flags is reported as an attack (class 1)."""
    def __init__(self, clf, gate): self.clf = clf; self.gate = gate; self.n_queries = 0
    def predict(self, X):
        X = np.ascontiguousarray(X, dtype=np.float32); self.n_queries += len(X); p = self.clf.predict(X)
        return np.where((p == 0) & self.gate.flag(X), 1, p)

class PipelineMargin:
    """the quantity the fully adaptive white-box attacker maximises: min(classifier margin gap, GATE_SCALE * (tau - error) / tau); > 0 <=> evades both layers.
    Returned as a 2-column array so that the existing gap_of(z) = z[:,0] - max(z[:,1:]) equals this score."""
    def __init__(self, clf_gpu, gate): self.m = Margin(clf_gpu); self.gate = gate
    def __call__(self, X):
        s = np.minimum(gap_of(self.m(X)), GATE_SCALE * (self.gate.tau - self.gate.score(X)) / self.gate.tau)
        return np.stack([s, np.zeros_like(s)], 1)
print('gate, pipeline oracle and joint-score margin defined')

## 4. Rows and gates

In [ ]:
rng0 = np.random.RandomState(0); pick = np.sort(np.concatenate([rng0.choice(np.flatnonzero(y_atk_test == c), N_WB_PER_CLASS, replace=False) for c in range(1, 6)]))
X_wb, y_wb = X_atk_test[pick], y_atk_test[pick]                           # NB5 / NB5b / NB5c rows
rng1 = np.random.RandomState(1); sub = np.sort(np.concatenate([rng1.choice(np.flatnonzero(y_wb == c), N_AW_PER_CLASS, replace=False) for c in range(1, 6)]))   # random subset (not the earliest rows)
X_aw, y_aw = X_wb[sub], y_wb[sub]
def pcs(ev, y): return np.array([ev[y == c].mean() for c in range(1, 6)])
def load_model(path, seed):
    gpu = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); gpu.load_model(path)
    cpu = xgb.XGBClassifier(); cpu.load_model(path); cpu.set_params(device='cpu', n_jobs=-1); return gpu, cpu
print(f'attack rows used by the white-box attackers: {len(X_aw)} ({N_AW_PER_CLASS} per class) | budgets {AW_BUDGETS} | PGD / PPO use all {len(X_atk_test):,} test attack rows')

In [ ]:
from sklearn.model_selection import KFold
rng2 = np.random.RandomState(3); sub2 = np.sort(np.concatenate([rng2.choice(np.flatnonzero(y_aw == c), N_ADAPT_PER_CLASS, replace=False) for c in range(1, 6)])); X_ad, y_ad = X_aw[sub2], y_aw[sub2]
print(f'rows attacked: {len(X_ad)} ({N_ADAPT_PER_CLASS} per class) | budgets {ADAPT_BUDGETS} | up to {ADAPT_ITERS} rounds | Layer 1: {L1_USE}')
benign_te = y_test == 0
def verdict(clf_cpu, gate, X):
    ev1 = clf_cpu.predict(X) == 0; fl = gate.flag(X); return ev1, fl, ev1 & ~fl
def load_ae(path, n_in=N_FEAT):
    ck = torch.load(path, map_location=DEVICE, weights_only=False); ms_ = []
    for st in ck['state']: m = AE(n_in).to(DEVICE); m.load_state_dict(st); m.eval(); ms_.append(m)
    return ms_, ck
def settings_for(seed):
    """name -> Gate. The NB6 gate at several percentiles of its out-of-fold error, plus the NB7 time-blocked gate."""
    ms_, ck = load_ae(f'ae_gate_{PIPELINE_VERSION}_seed{seed}.pt'); Xb = np.ascontiguousarray(X_train[y_train == 0]); oof = np.zeros(len(Xb))
    for k, (tr, ho) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=seed).split(Xb)): oof[ho] = Gate([ms_[k]], 0).score(Xb[ho])
    assert abs(np.percentile(oof, GATE_PCTL) - ck['tau']) <= 1e-4 * ck['tau'], 'NB6 threshold not reproduced'
    out = {f'rand_p{p:g}': Gate(ms_, np.percentile(oof, p)) for p in GATE_PCTLS_B}
    if INCLUDE_BLK:
        mb, ckb = load_ae(f'ae_gate7_all_blk_{PIPELINE_VERSION}_seed{seed}.pt'); out['blocked_p95'] = Gate(mb, ckb['tau'])
    return out

## 5. Per-seed run

In [ ]:
R = {}; t_all = time.time()
for seed in SEEDS:
    pf = f'attack3c_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = repr((L1_USE, GATE_PCTLS_B, INCLUDE_BLK, ADAPT_ITERS, ADAPT_BUDGETS, N_ADAPT_PER_CLASS, GATE_SCALE, K_CAND, seed, X_atk_test.shape, float(X_atk_test[:5].sum()), 'nb7b-v1'))
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: R[seed] = saved['R']; print(f'seed {seed}: result file found -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}', flush=True); r = {}; G = settings_for(seed)
    for mtag in L1_USE:
        gpu_m, cpu_m = load_model(LAYER1[mtag].format(v=PIPELINE_VERSION, s=seed), seed); TH = get_thresholds(gpu_m); mg = Margin(gpu_m); cl = cpu_m.predict(X_test)
        un = {}
        for bt, B in ADAPT_BUDGETS.items(): un[bt] = whitebox_trace(mg, X_ad, y_ad, CYBER_IDX, TH, B, verbose=False)[0]      # gate-unaware attack on Layer 1, once
        for gname, gate in G.items():
            sc = gate.score(X_test); fl_te = sc > gate.tau
            clean = {'tau': gate.tau, 'gate_fpr': float(fl_te[benign_te].mean()), 'gate_det': float(fl_te[~benign_te].mean()), 'gate_det_pc': np.array([fl_te[y_test == c].mean() for c in range(1, 6)]),
                     'pipeline_fpr': float((((cl != 0) | fl_te))[benign_te].mean()), 'pipeline_det': float((((cl != 0) | fl_te))[~benign_te].mean())}
            r[f'{mtag}|{gname}|clean'] = clean
            print(f'  [{mtag}|{gname}] tau {gate.tau:.5f} | gate FPR {clean["gate_fpr"]:.3f}, pipeline FPR {clean["pipeline_fpr"]:.3f}, gate detection {clean["gate_det"]:.3f} per class {np.round(clean["gate_det_pc"], 3).tolist()}', flush=True)
            pm = PipelineMargin(gpu_m, gate)
            for bt, B in ADAPT_BUDGETS.items():
                ev1u, flu, evpu = verdict(cpu_m, gate, un[bt])
                ts = time.time(); xa, ga, ie, st, tr = whitebox_trace(pm, X_ad, y_ad, CYBER_IDX, TH, B, max_iters=ADAPT_ITERS, verbose=False); ev1, fl, evp = verdict(cpu_m, gate, xa)
                r[f'{mtag}|{gname}|{bt}'] = {'l1_evasion': float(ev1u.mean()), 'unaware_pipeline': float(evpu.mean()), 'unaware_pc': pcs(evpu, y_ad), 'adaptive_pipeline': float(evp.mean()), 'adaptive_pc': pcs(evp, y_ad),
                                              'still_improving': float((~st & ~evp).mean()), 'still_improving_pc': np.array([(~st & ~evp)[y_ad == c].mean() for c in range(1, 6)]), 'search_vs_cpu_disagree': float(((ga > 0) != evp).mean()), 'rounds': len(tr)}
                q = r[f'{mtag}|{gname}|{bt}']
                print(f'      {bt} sigma: Layer 1 only {q["l1_evasion"]:.3f} | gate-unaware -> pipeline {q["unaware_pipeline"]:.3f} | FULLY ADAPTIVE -> pipeline {q["adaptive_pipeline"]:.3f} per class {np.round(q["adaptive_pc"], 3).tolist()} (still improving {q["still_improving"]:.3f}, rounds {q["rounds"]}) | {(time.time() - ts) / 60:.1f} min', flush=True)
        del mg, gpu_m, cpu_m; torch.cuda.empty_cache()
    R[seed] = r; joblib.dump({'fp': cfg_fp, 'R': r}, pf); print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min', flush=True)
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 6. Results

In [ ]:
atk_names = class_names[1:]
def ptest(a, b):
    d = np.asarray(a) - np.asarray(b)
    if len(d) < 2: return float('nan')
    return 1.0 if d.std(ddof=1) == 0 and d.mean() == 0 else (0.0 if d.std(ddof=1) == 0 else float(stats.ttest_rel(a, b).pvalue))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rk, i in enumerate(o): run = max(run, (len(p) - rk) * p[i]); adj[i] = min(1.0, run)
    return adj
ms = lambda x: f'{np.mean(x):.3f} +/- {(np.std(x, ddof=1) if len(x) > 1 else 0):.3f}'
GN = [k.split('|')[1] for k in R[SEEDS[0]] if k.endswith('|clean') and k.startswith(L1_USE[0])]; BT = list(ADAPT_BUDGETS)
pd.set_option('display.width', 280); pd.set_option('display.max_columns', 30)
for m in L1_USE:
    for bt in BT:
        print(f'\n7a. Layer 1 = {m}, {bt} sigma: protection against false alarms (mean +/- std over {len(SEEDS)} seeds)')
        rows = []
        for g in GN:
            c = [R[s][f'{m}|{g}|clean'] for s in SEEDS]; q = [R[s][f'{m}|{g}|{bt}'] for s in SEEDS]
            rows.append({'gate threshold': g, 'gate FPR': ms([x['gate_fpr'] for x in c]), 'pipeline FPR': ms([x['pipeline_fpr'] for x in c]), 'clean attack detection (pipeline)': ms([x['pipeline_det'] for x in c]),
                         'evades L1 only': ms([x['l1_evasion'] for x in q]), 'evades pipeline, gate-unaware': ms([x['unaware_pipeline'] for x in q]), 'evades pipeline, FULLY ADAPTIVE': ms([x['adaptive_pipeline'] for x in q]),
                         **{f'adaptive {n}': ms([x['adaptive_pc'][i] for x in q]) for i, n in enumerate(atk_names)}})
        T = pd.DataFrame(rows); print(T.to_string(index=False)); T.to_csv(f'attack3c_tradeoff_{m}_{bt}_{PIPELINE_VERSION}.csv', index=False)
        base = 'rand_p95'; print(f'   paired tests vs {base} (fully adaptive evasion, Holm over the other thresholds), per class where the class mean differs:')
        val = lambda s_, g_, i_: R[s_][f'{m}|{g_}|{bt}']['adaptive_pipeline'] if i_ == 0 else R[s_][f'{m}|{g_}|{bt}']['adaptive_pc'][i_ - 1]
        names = [g for g in GN if g != base]
        for i, n in enumerate(['all'] + atk_names):
            ref = np.array([val(s, base, i) for s in SEEDS]); cur = {g: np.array([val(s, g, i) for s in SEEDS]) for g in names}; ph = holm([ptest(cur[g], ref) for g in names])
            print(f'     {n:13s} {base} {ref.mean():.3f} |', {g: (round(float(cur[g].mean()), 3), round(float(p), 4)) for g, p in zip(names, ph)})
        print('   convergence: share of rows still improving when the search stopped:', {g: round(float(np.mean([R[s][f'{m}|{g}|{bt}']['still_improving'] for s in SEEDS])), 4) for g in GN}, '| search vs CPU disagreement:', {g: round(float(np.mean([R[s][f'{m}|{g}|{bt}']['search_vs_cpu_disagree'] for s in SEEDS])), 5) for g in GN})
        print('   rounds used (max over seeds):', {g: int(max(R[s][f'{m}|{g}|{bt}']['rounds'] for s in SEEDS)) for g in GN})

## 7. Figure and save

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5)); m = L1_USE[0]; bt = BT[0]
fp_ = np.array([np.mean([R[s][f'{m}|{g}|clean']['pipeline_fpr'] for s in SEEDS]) for g in GN])
for i, n in enumerate(atk_names):
    y = np.array([np.mean([R[s][f'{m}|{g}|{bt}']['adaptive_pc'][i] for s in SEEDS]) for g in GN]); e = np.array([np.std([R[s][f'{m}|{g}|{bt}']['adaptive_pc'][i] for s in SEEDS], ddof=1) for g in GN])
    ax[0].errorbar(fp_, y, yerr=e, marker='o', label=n, capsize=2)
for j, g in enumerate(GN): ax[0].annotate(g, (fp_[j], 0.01), fontsize=7, rotation=90, va='bottom')
ax[0].set_xlabel('pipeline false-positive rate on benign test rows'); ax[0].set_ylabel(f'fully adaptive pipeline evasion at {bt} sigma'); ax[0].set_title(f'Protection vs false alarms (Layer 1 = {m})'); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
w = 0.8 / len(GN)
for j, g in enumerate(GN): ax[1].bar(np.arange(5) + j * w, [np.mean([R[s][f'{m}|{g}|{bt}']['unaware_pc'][i] for s in SEEDS]) for i in range(5)], w, alpha=.45, color=f'C{j}')
for j, g in enumerate(GN): ax[1].bar(np.arange(5) + j * w, [np.mean([R[s][f'{m}|{g}|{bt}']['adaptive_pc'][i] for s in SEEDS]) for i in range(5)], w, fill=False, edgecolor=f'C{j}', linewidth=1.5, label=g)
ax[1].set_xticks(np.arange(5) + 0.4 - w / 2); ax[1].set_xticklabels(atk_names); ax[1].set_title('Filled: gate-unaware attacker, outline: fully adaptive'); ax[1].legend(fontsize=7); ax[1].grid(alpha=.3, axis='y')
plt.tight_layout(); plt.savefig(f'attack3c_tradeoff_{PIPELINE_VERSION}.png', dpi=150); plt.show()
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'per_seed': {s: R[s] for s in SEEDS}, 'config': {'L1_USE': L1_USE, 'GATE_PCTLS_B': GATE_PCTLS_B, 'INCLUDE_BLK': INCLUDE_BLK, 'ADAPT_ITERS': ADAPT_ITERS, 'ADAPT_BUDGETS': ADAPT_BUDGETS, 'N_ADAPT_PER_CLASS': N_ADAPT_PER_CLASS}}, f'attack3c_results_{PIPELINE_VERSION}.pkl')
json.dump({'notebook': 'NB7b', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'xgboost': xgb.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb7b_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'attack3c_results_{PIPELINE_VERSION}.pkl', f'attack3c_tradeoff_{PIPELINE_VERSION}.png', f'nb7b_run_manifest_{PIPELINE_VERSION}.json'] + [f'attack3c_tradeoff_{m}_{bt}_{PIPELINE_VERSION}.csv' for m in L1_USE for bt in BT]: print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)